# Environment Requirements
To run this repository successfully, the following environment is required:
- **GPU**: NVIDIA A100 with 80GB High RAM
- **Python**: 3.10 or higher
- **CUDA**: 12.1 or higher
- **PyTorch**: 2.1 or higher

In [ ]:
# --- HUGGING FACE TOKEN INSTRUCTIONS ---
# 1. Go to https://huggingface.co/settings/tokens
# 2. Click on "New token" to create a token (Read access is usually sufficient).
# 3. Copy the token and replace the placeholder string below.
# ---------------------------------------

HF_TOKEN = "hf_xxx_your_token_here"

In [ ]:
# Perform the login
!hf auth login --token HF_TOKEN
print("Hugging Face login successful!")

In [2]:
import sys
import os
import subprocess
import multiprocessing
import gc
import torch

In [3]:
# Set the start method globally once
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

def cleanup_vllm_and_gpu():
    """
    Resets the vLLM singleton and clears GPU memory cache.
    Call this after every benchmark run to ensure stability.
    """
    # Reset the LLM singleton if it exists
    try:
        import utils.llm as _llm_mod
        if _llm_mod._backend_instance is not None:
            print("[Cleanup] Resetting stale LLM singleton …")
            _llm_mod._backend_instance = None
    except Exception as e:
        print(f"[Cleanup] LLM reset skipped or failed: {e}")

    # Clear GPU memory
    try:
        gc.collect()
        torch.cuda.empty_cache()
        free_mem = torch.cuda.mem_get_info()[0] / 1e9
        print(f"[Cleanup] GPU memory freed. Available: {free_mem:.1f} GB")
    except Exception as e:
        print(f"[Cleanup] Memory clearing failed: {e}")

    print("[Cleanup] Ready for next run.")

def setup_git_repo(repo_url, base_path="/content"):
    """
    Clones a git repository or pulls updates if it already exists.

    Args:
        repo_url (str): The HTTPS URL of the GitHub repository.
        base_path (str): The local directory where the repo should live.
    """
    # Ensure we are in the correct base directory
    os.chdir(base_path)

    # Extract the repository name from the URL (e.g., 'raise-26')
    repo_dir = repo_url.split("/")[-1].replace(".git", "")
    repo_path = os.path.join(base_path, repo_dir)

    if os.path.exists(repo_path):
        print(f"Directory '{repo_dir}' exists. Pulling latest changes...")
        # Run git pull inside the specific directory
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    else:
        print(f"Directory '{repo_dir}' not found. Cloning...")
        subprocess.run(["git", "clone", repo_url], check=True)

    return repo_path

def check_environment():
    # Check Hugging Face Token
    assert HF_TOKEN != "hf_xxx_your_token_here" and HF_TOKEN.startswith("hf_"), (
        "Please replace the placeholder with a valid Hugging Face token starting with 'hf_'."
    )

    # Check Python version
    assert sys.version_info >= (3, 10), f"Python version must be >= 3.10. Found {sys.version.split()[0]}"

    # Check PyTorch version
    torch_version = torch.__version__
    assert int(torch_version.split('.')[0]) >= 2, f"PyTorch version must be >= 2.0. Found {torch_version}"

    # Check CUDA availability
    assert torch.cuda.is_available(), "CUDA is not available. Please enable a GPU runtime."

    # Check CUDA version
    cuda_version = torch.version.cuda
    assert cuda_version is not None and float('.'.join(cuda_version.split('.')[:2])) >= 12.1, f"CUDA version must be >= 12.1. Found {cuda_version}"

    # Check GPU type and memory
    device_name = torch.cuda.get_device_name(0)
    assert "A100" in device_name, f"An A100 GPU is required. Found: {device_name}"

    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    assert total_memory_gb >= 79, f"GPU memory must be at least 80GB. Found: {total_memory_gb:.1f}GB"

    print("All environment checks passed successfully! ✅")

In [4]:
check_environment()

All environment checks passed successfully! ✅


In [ ]:
%%capture
%pip install -U vllm --extra-index-url https://wheels.vllm.ai/nightly
%pip install chromadb flask-cors flask-socketio "git+https://github.com/huggingface/transformers.git"
%pip install "https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.0/flash_attn-2.8.3+cu128torch2.10-cp312-cp312-linux_x86_64.whl"

Looking in indexes: https://pypi.org/simple, https://wheels.vllm.ai/nightly
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.6.0.dev0
    Uninstalling transformers-5.6.0.dev0:
      Successfully uninstalled transformers-5.6.0.dev0


  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-3fqqz630
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-3fqqz630
  Resolved https://github.com/huggingface/transformers.git to commit a553395766001116a719c82870171f8d6b458c98
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached chromadb-1.5.7-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached huggingface_hub-1.10.1-py3-none-any.whl.metadata (14 kB)
Using cached chromadb-1.5.7-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (23.0 MB)
Using cached huggingface_hub-1.10.1-py3-none-any.whl (642 kB)
  Created wheel for transformers: filename=transformers-5.6.0.dev0-py3-none-any.whl size=11381798 sha256=0d0d708c5e1171c6d8d31fc02bf5c522b31e5b7e97c8295f077817a2c23dc76f
  Stored in directory: /tmp/pip

In [ ]:
setup_git_repo("https://github.com/pandevim/TriMem.git")
%cd /content/TriMem

In [ ]:
%run run_benchmark.py --agent baseline --tasks 1

In [ ]:
%run run_benchmark.py --agent rag --tasks 1

In [ ]:
%run run_benchmark.py --agent visual_bus --tasks 1


[INIT] Creating visual_bus agent …
[LLM] Loading Qwen/Qwen3.5-35B-A3B via vLLM …
[LLM] Initializing vLLM engine (model=Qwen/Qwen3.5-35B-A3B, gpu_mem=0.9, max_len=16384) …
INFO 04-10 22:58:37 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 16384, 'disable_log_stats': True, 'model': 'Qwen/Qwen3.5-35B-A3B'}
INFO 04-10 22:58:38 [model.py:549] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 04-10 22:58:38 [model.py:1678] Using max model len 16384
INFO 04-10 22:58:38 [config.py:281] Setting attention block size to 1056 tokens to ensure that attention page size is >= mamba page size.
INFO 04-10 22:58:38 [config.py:312] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
[LLM] vLLM engine ready in 117.8s
[LLM] Ready.
[INIT] Loading 1 tasks …

############################################################
  BENCHMARK: VISUAL_BUS agent | 1 tasks
#####################################################


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 0.00/79.3M [00:00<?, ?iB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 17.0k/79.3M [00:00<17:38, 78.6kiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 51.0k/79.3M [00:00<11:05, 125kiB/s] 
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 119k/79.3M [00:00<06:32, 212kiB/s] 
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 272k/79.3M [00:00<03:23, 408kiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   1%|          | 561k/79.3M [00:01<01:51, 744kiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   1%|▏         | 1.15M/79.3M [00:01<00:56, 1.45MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   3%|▎         | 2.32M/79.3M [00:01<00:28, 2.80MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   6%|▌         | 4.48M/

[LLM] Generating (vLLM, 2 messages) … 

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

done in 2399ms (in=1605, out=306)
  [LLM] access compliance_dashboard
  ✗ access compliance_dashboard              → Command executed but returned no results or failed.
  [Turn 35/60] (elapsed: 786.1s) Thinking … [VisualBus] render=272ms  ocr=28133ms  compressed=4412 chars
[LLM] Generating (vLLM, 2 messages) … 

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

done in 2488ms (in=1573, out=319)
  [LLM] upload invoice_1 to compliance_dashboard
  ✓ upload invoice_1 to compliance_dashboard → Data transfer complete. invoice_1 uploaded.

Audit task comp
Result: SUCCESS | Turns: 36 | Duration: 817.5s | Tokens: 58155 | Syntax errors: 6
[PROGRESS] 1/1 tasks done | Elapsed: 817.5s | Est. remaining: 0.0s
Results saved to logs/visual_bus_1775862852.json
[SAVE] Results written to logs/visual_bus_1775862852.json

  SUMMARY: VISUAL_BUS (817.5s total)
  agent_type                    : visual_bus
  num_tasks                     : 1
  success_rate                  : 1.0
  avg_turns                     : 36.0
  avg_tokens                    : 58155.0
  total_syntactic_errors        : 6
  total_spatial_hallucinations  : 19
  total_cost_usd                : 0.0

